In [0]:
df = spark.read.parquet("dbfs:/mnt/light/data/world_bank_global_postings/version=3/2025/02/26/00_26_56/*.parquet")

In [0]:
from pyspark.sql.functions import col, count, month, year, to_date, concat, lit, when, last_day, lpad, sum as _sum

# Filter for US & jobs posted after 2017-01-01
df_us = df.filter((col('nation_name') == "United States") & (col('posted') >= "2021-01-01") & (col('posted') < "2023-01-01"))

# Filter to only include naics code 21 for grey industry jobs
df_us = df_us.filter((col('naics2') == 21))

# Compile the DataFrame with necessary transformations
new_df = (
    df_us.withColumn("month", lpad(month(col("posted")).cast("string"), 2, "0"))  # Ensures 01, 02, ..., 12
    .withColumn("year", year(col("posted")))
    .groupBy("year", "month", "laa_admin_area_2", "min_years_experience", "isco_08_1_name")
    .agg(
        count(when(col("employment_type") == 1, True)).alias("full_time_count"),
        count(when(col("employment_type") == 2, True)).alias("part_time_count"),
        count("*").alias("total_count")
    )
    .withColumnRenamed("laa_admin_area_2", "county_id")
    .withColumnRenamed("isco_08_1_name", "career_type")
    .withColumn("start_date", to_date(concat(col("year"), lit("-"), col("month"), lit("-01")), "yyyy-MM-dd"))
    .withColumn("end_date", last_day(col("start_date")))  # Ensure correct last day
)

# Display the final compiled DataFrame
display(new_df)

year,month,county_id,min_years_experience,career_type,full_time_count,part_time_count,total_count,start_date,end_date
2022,10,US54039,5,Technicians and Associate Professionals,3,0,3,2022-10-01,2022-10-31
2022,03,US45055,null,Elementary Occupations,1,0,1,2022-03-01,2022-03-31
2022,11,US12021,1,Professionals,8,6,14,2022-11-01,2022-11-30
2022,08,US47037,null,"Plant and Machine Operators, and Assemblers",1,0,1,2022-08-01,2022-08-31
2021,06,US48439,6,Professionals,1,0,1,2021-06-01,2021-06-30
2021,10,US17031,5,Professionals,2,0,2,2021-10-01,2021-10-31
2022,03,US48201,3,Professionals,42,0,44,2022-03-01,2022-03-31
2021,10,US13067,5,Professionals,5,0,5,2021-10-01,2021-10-31
2021,10,US4012,null,Technicians and Associate Professionals,1,0,1,2021-10-01,2021-10-31
2021,10,US48453,1,"Plant and Machine Operators, and Assemblers",4,0,4,2021-10-01,2021-10-31


In [0]:
output_path = "/Volumes/prd_datascience_onlinejobpostings/volumes/onlinejobpostings/joep_keuzenkamp/grey_vacancies_data_2021_22"

# Append data as CSV (instead of Parquet)
new_df.write.format("csv") \
    .mode("overwrite") \
    .option("header", "true") \
    .option("sep", ",") \
    .save(output_path)